# Moirai2 + GraphSAGE: VN30 Realized Volatility Prediction

End-to-end pipeline:
1. Load daily OHLCV data for 30 VN30 stocks + VNINDEX
2. Compute RV₂₀ labels and Moirai2-small embeddings
3. Build correlation+sector graph (monthly snapshots)
4. Train VolatilityGNN (GraphSAGE + MLP head) walk-forward
5. Train MLP + Moirai2 ablation baseline (no graph)
6. Evaluate against GARCH(1,1), HAR-RV, LSTM, MLP+Moirai2
7. Report full metric table with DM tests

**Key ablation:** MLP+Moirai2 vs GNN — if GNN > MLP, the graph structure contributes beyond the embeddings alone.

In [ ]:
import os, sys
sys.path.insert(0, '..')
os.environ.setdefault('HF_HOME', r'D:\hf_cache')

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

# Project imports
from src.volatility_labels import load_close_prices, compute_log_returns, compute_rv
from src.embed_extractor import Moirai2Embedder
from gnn.build_graph import build_graph, VN30_TICKERS, ALL_NODES
from gnn.model import VolatilityGNN
from gnn.train import train_walkforward, extract_embeddings
from baselines.garch_baseline import run_garch_baseline
from baselines.har_rv_baseline import run_har_baseline
from baselines.lstm_baseline import run_lstm_baseline
from baselines.mlp_baseline import train_mlp_walkforward, run_mlp_inference, VolatilityMLP
from evaluation.metrics import compare_models, diebold_mariano

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

PRICES_DIR = f"../{cfg['data']['prices_dir']}"
TRAIN_END  = pd.Timestamp(cfg['data']['train_end'])
TEST_START = pd.Timestamp(cfg['data']['test_start'])
HORIZON    = cfg['model']['horizon']
RESULTS    = '../results'
os.makedirs(RESULTS, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Train end: {TRAIN_END.date()}   Test start: {TEST_START.date()}')

## 1. Data Overview

In [ ]:
close   = load_close_prices(PRICES_DIR, tickers=VN30_TICKERS + ['VNINDEX'])
log_ret = compute_log_returns(close)
rv_all  = compute_rv(close[VN30_TICKERS], h=HORIZON)

print(f'Close prices:  {close.shape}  ({close.index[0].date()} → {close.index[-1].date()})')
print(f'Log returns:   {log_ret.shape}')
print(f'RV₂₀ labels:   {rv_all.shape}')

train_rv = rv_all[rv_all.index <= TRAIN_END]
test_rv  = rv_all[rv_all.index >= TEST_START]
print(f'\nTrain RV windows: {train_rv.notna().all(axis=1).sum()}')
print(f'Test  RV windows: {test_rv.notna().all(axis=1).sum()}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

norm = close / close.iloc[0]
for col in VN30_TICKERS:
    axes[0].plot(norm.index, norm[col], alpha=0.3, lw=0.8)
axes[0].plot(norm.index, norm['VNINDEX'], 'k-', lw=1.5, label='VNINDEX')
axes[0].axvline(TRAIN_END, color='red', ls='--', lw=1.2, label='Train end')
axes[0].set_title('VN30 Normalised Prices (base=1)')
axes[0].legend()

sample_tickers = ['VCB', 'HPG', 'FPT', 'MWG']
for t in sample_tickers:
    axes[1].plot(rv_all.index, rv_all[t], alpha=0.7, lw=0.9, label=t)
axes[1].axvline(TRAIN_END, color='red', ls='--', lw=1.2)
axes[1].set_title('RV₂₀ (realized volatility, 20-day horizon)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{RESULTS}/01_data_overview.png', dpi=120)
plt.show()

## 2. Graph Snapshot

In [ ]:
graph = build_graph(
    log_ret, end_date=TRAIN_END,
    corr_window=cfg['model']['corr_window'],
    corr_threshold=cfg['model']['corr_threshold'],
)
n_edges = graph.edge_index.shape[1]
hub_e   = int((graph.edge_index[0] == 0).sum())
print(f'Graph at {TRAIN_END.date()}:')
print(f'  Nodes: {graph.num_nodes},  Total edges: {n_edges}')
print(f'  Hub edges (VNINDEX->stocks): {hub_e},  Stock-stock: {n_edges - hub_e}')
print(f'  Loss mask: {graph.loss_mask.sum().item()}/31 in loss')

## 3. Train GNN (Walk-forward)

In [ ]:
metrics_df = train_walkforward(cfg, results_dir=RESULTS)
print(f'Windows trained: {len(metrics_df)}')
print(metrics_df.tail(5).to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(metrics_df['window'], metrics_df['train_loss'], marker='o', ms=3)
plt.xlabel('Window')
plt.ylabel('MSE Loss')
plt.title('GNN Train Loss per Walk-forward Window')
plt.tight_layout()
plt.savefig(f'{RESULTS}/02_train_loss.png', dpi=120)
plt.show()

## 3b. Train MLP + Moirai2 Ablation Baseline (no graph)

Identical training setup as GNN — same embeddings, same walk-forward windows, same MSE loss — but **no graph structure**.  
This ablation answers: **does the graph add value beyond Moirai2 embeddings?**

In [ ]:
mlp_metrics_df = train_mlp_walkforward(cfg, results_dir=RESULTS)
print(f'MLP windows trained: {len(mlp_metrics_df)}')
print(mlp_metrics_df.tail(5).to_string(index=False))

## 4. GNN Test-set Inference

In [ ]:
# Load best GNN checkpoint
gnn_model = VolatilityGNN(
    in_dim=Moirai2Embedder.D_MODEL,
    hidden=cfg['model']['gnn_hidden'],
    mlp_hidden=cfg['model']['mlp_hidden'],
    dropout=cfg['model']['dropout'],
).to(device)
gnn_model.load_state_dict(torch.load(f'{RESULTS}/best_gnn.pt', map_location=device))
gnn_model.eval()

embedder = Moirai2Embedder(
    size='small',
    context_length=cfg['model']['context_length'],
    patch_size=cfg['model'].get('patch_size', 32),
)

test_dates = rv_all.index[
    (rv_all.index >= TEST_START) & (~rv_all.isna().all(axis=1))
]
print(f'Test dates: {len(test_dates)}  ({test_dates[0].date()} → {test_dates[-1].date()})')

In [ ]:
gnn_preds = {t: {} for t in VN30_TICKERS}

with torch.no_grad():
    for date in test_dates:
        graph_t = build_graph(
            log_ret, end_date=date,
            corr_window=cfg['model']['corr_window'],
            corr_threshold=cfg['model']['corr_threshold'],
        )
        node_feats = extract_embeddings(
            embedder, log_ret, date, cfg['model']['context_length']
        ).to(device)
        pred = gnn_model(node_feats, graph_t.edge_index.to(device)).cpu().numpy().ravel()
        for i, ticker in enumerate(VN30_TICKERS):
            gnn_preds[ticker][date] = max(float(pred[i + 1]), 0.0)

gnn_pred_df = pd.DataFrame({t: pd.Series(gnn_preds[t]) for t in VN30_TICKERS})
print(f'GNN predictions: {gnn_pred_df.shape}')

## 4b. MLP + Moirai2 Test-set Inference

In [ ]:
mlp_preds_dict = run_mlp_inference(
    cfg,
    checkpoint_path=f'{RESULTS}/best_mlp.pt',
    results_dir=RESULTS,
)
mlp_pred_df = pd.DataFrame(mlp_preds_dict)
print(f'MLP+Moirai2 predictions: {mlp_pred_df.shape}')

## 5. Statistical Baselines

In [ ]:
print('Running GARCH(1,1)...')
garch_df = pd.DataFrame(run_garch_baseline(
    PRICES_DIR, train_end=cfg['data']['train_end'],
    test_start=cfg['data']['test_start'], horizon=HORIZON,
))
print(f'  GARCH: {garch_df.shape}')

print('Running HAR-RV...')
har_df = pd.DataFrame(run_har_baseline(
    PRICES_DIR, train_end=cfg['data']['train_end'],
    test_start=cfg['data']['test_start'], horizon=HORIZON,
))
print(f'  HAR:   {har_df.shape}')

print('Running LSTM...')
lstm_df = pd.DataFrame(run_lstm_baseline(
    PRICES_DIR, train_end=cfg['data']['train_end'],
    test_start=cfg['data']['test_start'], horizon=HORIZON,
))
print(f'  LSTM:  {lstm_df.shape}')

## 6. Metric Table (pooled across all 30 stocks)

In [ ]:
def pool_predictions(pred_df, true_df):
    common = pred_df.index.intersection(true_df.index)
    y_true = true_df.loc[common].values.ravel()
    y_pred = pred_df.loc[common].values.ravel()
    valid  = ~(np.isnan(y_true) | np.isnan(y_pred))
    return y_true[valid], y_pred[valid]

yt_gnn,  yp_gnn  = pool_predictions(gnn_pred_df,  rv_all)
yt_mlp,  yp_mlp  = pool_predictions(mlp_pred_df,  rv_all)
yt_garch,yp_garch = pool_predictions(garch_df,     rv_all)
yt_har,  yp_har  = pool_predictions(har_df,        rv_all)
yt_lstm, yp_lstm  = pool_predictions(lstm_df,       rv_all)

yt_base = yt_gnn
n = len(yt_base)

def _align(yp, n):
    if len(yp) >= n: return yp[:n]
    return np.concatenate([yp, np.full(n - len(yp), np.nan)])

results_table = compare_models(
    yt_base,
    {
        'GNN (ours)':        yp_gnn,
        'MLP + Moirai2':     _align(yp_mlp,  n),
        'GARCH(1,1)':        _align(yp_garch, n),
        'HAR-RV':            _align(yp_har,  n),
        'LSTM':              _align(yp_lstm, n),
    },
    dm_reference='GNN (ours)',
    dm_loss=cfg['evaluation']['dm_loss'],
)

display_cols = ['MAE', 'RMSE', 'R2', 'QLIKE', 'Pearson_r', 'DM_stat', 'DM_pval']
print('\n=== Volatility Prediction Results (2026 test set, pooled) ===\n')
print(results_table[display_cols].round(4).to_string())

results_table.to_csv(f'{RESULTS}/model_comparison.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'QLIKE']):
    vals   = results_table[metric]
    colors = ['#1565C0' if idx == 'GNN (ours)'
              else '#42A5F5' if idx == 'MLP + Moirai2'
              else '#90CAF9' for idx in vals.index]
    bars = ax.bar(vals.index, vals, color=colors)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=20)
    ax.bar_label(bars, fmt='%.4f', fontsize=8)

plt.suptitle('Model Comparison — 2026 Test Set\n(dark blue=GNN, mid=MLP+Moirai2, light=statistical baselines)',
             fontsize=11, y=1.04)
plt.tight_layout()
plt.savefig(f'{RESULTS}/03_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Scatter: Predicted vs Realized (GNN, one stock)

In [ ]:
ticker = 'VCB'
rv_t = rv_all[ticker].loc[gnn_pred_df.index].dropna()
gnn_t = gnn_pred_df[ticker].reindex(rv_t.index).dropna()
mlp_t = mlp_pred_df[ticker].reindex(rv_t.index).dropna() if ticker in mlp_pred_df.columns else None
rv_plot = rv_t.reindex(gnn_t.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(rv_plot.index, rv_plot.values, 'k-', label='Realized RV₂₀', lw=1.2)
axes[0].plot(gnn_t.index, gnn_t.values, '--', label='GNN pred', lw=1.2)
if mlp_t is not None:
    axes[0].plot(mlp_t.reindex(gnn_t.index).index, mlp_t.reindex(gnn_t.index).values,
                 ':', label='MLP pred', lw=1.0, alpha=0.8)
axes[0].set_title(f'{ticker} — Test period')
axes[0].legend()

axes[1].scatter(rv_plot.values, gnn_t.values, s=15, alpha=0.6, label='GNN')
lim = max(rv_plot.max(), gnn_t.max()) * 1.05
axes[1].plot([0, lim], [0, lim], 'r--', lw=1)
axes[1].set_xlabel('Realized RV')
axes[1].set_ylabel('Predicted RV')
axes[1].set_title(f'{ticker} — Predicted vs Realized (GNN)')

plt.tight_layout()
plt.savefig(f'{RESULTS}/04_scatter_{ticker}.png', dpi=120)
plt.show()

## 8. Per-stock R² heatmap

In [ ]:
from evaluation.metrics import r2_score as ev_r2

r2_table = pd.DataFrame(index=VN30_TICKERS,
                        columns=['GNN', 'MLP+M2', 'HAR', 'GARCH', 'LSTM'])
model_dfs = [
    ('GNN',    gnn_pred_df),
    ('MLP+M2', mlp_pred_df),
    ('HAR',    har_df),
    ('GARCH',  garch_df),
    ('LSTM',   lstm_df),
]
test_idx = rv_all.index[rv_all.index >= TEST_START]

for ticker in VN30_TICKERS:
    rv_t = rv_all[ticker].reindex(test_idx).dropna()
    for model_name, pred_df in model_dfs:
        if ticker not in pred_df.columns: continue
        shared = rv_t.index.intersection(pred_df[ticker].dropna().index)
        if len(shared) > 5:
            r2_table.loc[ticker, model_name] = ev_r2(
                rv_t.loc[shared].values, pred_df[ticker].loc[shared].values
            )

r2_table = r2_table.astype(float)

plt.figure(figsize=(8, 10))
sns.heatmap(r2_table, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-0.5, vmax=0.8, linewidths=0.5)
plt.title('Per-stock R² — 2026 Test Set')
plt.tight_layout()
plt.savefig(f'{RESULTS}/05_r2_heatmap.png', dpi=120)
plt.show()

print('\nMean R² across 30 stocks:')
print(r2_table.mean().round(3).to_string())

## 9. Summary

In [ ]:
print('=== Final Results Summary ===')
print(results_table[['MAE', 'RMSE', 'R2', 'QLIKE']].round(5).to_string())

# Key ablation finding
if 'MLP + Moirai2' in results_table.index and 'GNN (ours)' in results_table.index:
    gnn_mae = results_table.loc['GNN (ours)', 'MAE']
    mlp_mae = results_table.loc['MLP + Moirai2', 'MAE']
    delta   = (mlp_mae - gnn_mae) / mlp_mae * 100
    print(f'\nGraph contribution (MAE): GNN {gnn_mae:.5f} vs MLP+M2 {mlp_mae:.5f} = {delta:+.1f}%')
    if delta > 0:
        print('  => Graph HELPS: GNN outperforms MLP+Moirai2')
    else:
        print('  => Graph does NOT help in this setting')

print(f'\nAll results saved to {RESULTS}/')